In [1]:
import sys
import os
import torch 
import torchaudio
import numpy as np 

# Add parent directory to sys.path
parent_dir = os.path.abspath("..")
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from model import ModelArgs, TextAudioLMFromPretrained
from contextlib import nullcontext

torch.backends.cuda.matmul.allow_tf32 = True  # allow tf32 on matmul
torch.backends.cudnn.allow_tf32 = True  # allow tf32 on cudnn
device_type = "cuda" if torch.cuda.is_available() else "cpu"  # for later use in torch.autocast
device = torch.device(device_type)
# note: float16 data type will automatically use a GradScaler
dtype = "bfloat16"
ptdtype = {"float32": torch.float32, "bfloat16": torch.bfloat16, "float16": torch.float16}[dtype]
ctx = nullcontext() if device_type == "cpu" else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

# Setup

In [2]:
def save_audio(waveform, filename, sr=16000):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    torchaudio.save(filename, waveform, sr)

In [ ]:
# load pre-trained ckpt
from model import load_ckpt
model = load_ckpt("Anon-SmolTolk/SmolTolk-2B", "model_ckpt.pt", device=device)

In [4]:
from transformers import AutoTokenizer
# load txt tokenizer
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M")

In [ ]:
# load audio vocoder
from textless.vocoders.hifigan.vocoder import CodeHiFiGANVocoder
import IPython
from IPython.display import display, Audio

vocab_size = 500
vocoder = CodeHiFiGANVocoder.by_name(
    dense_model_name = "mhubert-base-25hz",
    quantizer_model_name = "kmeans",
    vocab_size = vocab_size
).eval()

def play_translation(input):
    # input = torch.from_numpy(input)
    wav = vocoder(input, dur_prediction = True)
    return wav.squeeze().cpu().numpy()


# Inference

Text to Text generation

In [ ]:
prompt = "Life is"
# Fake codebook dim: B, T -> B, 1, T
prompt = tokenizer.encode(prompt, return_tensors="pt").unsqueeze(0)
audio_input_masks = torch.zeros(prompt.shape[0], prompt.shape[-1]).bool()
with ctx, torch.no_grad():
    new_tokens, _ = model.generate(
        prompt.to(device), 
        audio_input_masks.to(device), 
        250, 
        temperature=0.4,
        top_k=50
    )
print(tokenizer.decode(new_tokens.view(-1)))

Uncond. Speech to Speech generation

In [ ]:
# Speech -> Speech
# Uncond. generation
# `500` is the eos/bos speech token.
prompt = torch.tensor([[[500]]], dtype=torch.long, device=device)
audio_input_masks = torch.ones(prompt.shape[0], prompt.shape[-1]).bool()
max_new_tokens = 300
temp = 0.8
top_k = 50

with ctx, torch.no_grad():
    new_tokens, _ = model.generate(
        prompt.to(device), 
        audio_input_masks.to(device), 
        max_new_tokens, 
        temperature=temp,
        top_k=top_k
    )

# we skip bos from vocoder
display(Audio(play_translation(new_tokens[..., 1:].cpu()), rate=16000))

Cond. Speech to Speech generation

In [ ]:
max_new_tokens = 300
temp = 0.8
top_k = 50

# We load a sample from the test-set of LibriSpeech
eval_data_dir = os.path.join('data', 'ls960', 'mhubert25hzl11')
eval_data = np.memmap(os.path.join(eval_data_dir, 'test-clean.bin'), dtype=np.uint16, mode='r')
eval_lens = np.memmap(os.path.join(eval_data_dir, 'test-clean.len'), dtype=np.uint16, mode='r')
eval_data = np.split(eval_data, np.cumsum(eval_lens)[:-1])
# sample random sample and only take half of the audio
idx = torch.randint(0, len(eval_data), (1,))
prompt = eval_data[idx][:len(eval_data[idx])//2]

prompt = torch.from_numpy(prompt).to(torch.long).to(device).unsqueeze(0).unsqueeze(0)
bos = torch.tensor([[[500]]], dtype=torch.long, device=device)
prompt = torch.cat((bos, prompt), -1)
audio_input_masks = torch.ones(prompt.shape[0], prompt.shape[-1]).bool()

# skip first token bos
display(Audio(play_translation(prompt[..., 1:].cpu()), rate=16000))

with ctx, torch.no_grad():
    new_tokens, _ = model.generate(
        prompt.to(device), 
        audio_input_masks.to(device), 
        max_new_tokens, 
        temperature=temp,
        top_k=top_k
    )

# skip the bos/eos token
mask = (new_tokens == 500)
display(Audio(play_translation(new_tokens[~mask].cpu()), rate=16000))

Cross-modal Text to Speech generation

In [ ]:
# Text -> Speech
prompt = tokenizer("the president of the united states is", return_tensors="pt").input_ids.to(device).unsqueeze(0)
# add bos/swt token
bos = torch.tensor([[[0]]], dtype=torch.long, device=device)
prompt = torch.cat((bos, prompt, torch.tensor([[[model.params.swt_token]]], device=device)), dim=-1)
audio_input_masks = torch.zeros_like(prompt).bool()[0]

max_new_tokens = 300
temp = 0.8
top_k = 50

with ctx, torch.no_grad():
    new_tokens, masks = model.generate(
        prompt.to(device), 
        audio_input_masks.to(device), 
        max_new_tokens, 
        temperature=temp,
        top_k=top_k
    )

# we skip swt token 
display(tokenizer.decode(prompt[..., 1:][0, 0]))
audio_tokens = new_tokens[masks[1].unsqueeze(0)][1:]
mask = (audio_tokens == 500)
display(Audio(play_translation(audio_tokens[~mask].cpu()), rate=16000))

Cross-modal Speech to Text generation

In [ ]:
# We load a sample from the test-set of LibriSpeech
eval_data_dir = os.path.join('data', 'ls960', 'mhubert25hzl11')
eval_data = np.memmap(os.path.join(eval_data_dir, 'test-clean.bin'), dtype=np.uint16, mode='r')
eval_lens = np.memmap(os.path.join(eval_data_dir, 'test-clean.len'), dtype=np.uint16, mode='r')
eval_data = np.split(eval_data, np.cumsum(eval_lens)[:-1])
# sample random sample and only take half of the audio
idx = torch.randint(0, len(eval_data), (1,))
prompt = eval_data[idx][:len(eval_data[idx])//2]

audio_prompt = torch.from_numpy(prompt).to(torch.long).to(device).unsqueeze(0).unsqueeze(0)
# add swt token
bos = torch.tensor([[[500]]], dtype=torch.long, device=device)
prompt = torch.cat((bos, audio_prompt, torch.tensor([[[model.params.swt_token]]], device=device)), dim=-1)
audio_input_masks = torch.ones_like(prompt).bool()[0]
audio_input_masks[..., -1] = False

max_new_tokens = 300
temp = 0.8
top_k = 50

with ctx, torch.no_grad():
    new_tokens, masks = model.generate(
        prompt.to(device), 
        audio_input_masks.to(device), 
        max_new_tokens, 
        temperature=temp,
        top_k=top_k
    )

# we skip swt token 
display(Audio(play_translation(audio_prompt.cpu()), rate=16000))
display(tokenizer.decode(new_tokens[~masks[0].unsqueeze(0)][1:], skip_special_tokens=True))